# Phase B-5 — Emotion Code & Steering Results

Llama-3.1-8B-Instruct, 4000 contrastive pairs, 8 Plutchik categories, hook layers `[13, 16, 19, 22]`.

Artifacts visualized:
- `data/emotion_code/caa.pt` — CAA direction per (category, layer)
- `data/emotion_code/basis.pt` — NMF / PCA emotion-code basis at layer 16
- `data/emotion_code/vad_mapping.pt` — Ridge V/A/D readout at layer 19
- `experiments/results/*` — layer sweep, shift accuracy, monotonicity, perplexity guardrail, generations cache

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path('..').resolve()
EC = ROOT / 'data' / 'emotion_code'
RES = ROOT / 'experiments' / 'results'

plt.rcParams.update({
    'figure.dpi': 110,
    'figure.figsize': (7, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 10,
})

PLUTCHIK_COLORS = {
    'joy':          '#F4C53D',
    'trust':        '#8FCB6F',
    'fear':         '#3FAE6A',
    'surprise':     '#36B5C6',
    'sadness':      '#4F7BC7',
    'disgust':      '#9367B5',
    'anger':        '#D9462E',
    'anticipation': '#E7873B',
}
PLUTCHIK_ORDER = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']

## 1. CAA direction — norm by (category × layer)
How strong is the contrastive direction at each hook layer?

In [ ]:
caa = torch.load(EC / 'caa.pt', weights_only=False, map_location='cpu')
cats = caa['categories']
layers = caa['layers']
vectors = caa['vectors']  # [C, L, D]
norms = vectors.norm(dim=-1).numpy()  # [C, L]

fig, ax = plt.subplots(figsize=(5.5, 4))
im = ax.imshow(norms, cmap='magma', aspect='auto')
ax.set_xticks(range(len(layers)), layers)
ax.set_yticks(range(len(cats)), cats)
ax.set_xlabel('hook layer')
ax.set_title('||v_{cat,layer}|| (CAA direction norm)')
for i in range(len(cats)):
    for j in range(len(layers)):
        ax.text(j, i, f'{norms[i,j]:.2f}', ha='center', va='center',
                color='white' if norms[i,j] < norms.max()*0.6 else 'black', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout()

## 2. CAA inter-category cosine similarity (layer 16)
Plutchik wheel structure: opposites should be anti-correlated, neighbours positive.

In [ ]:
INJECT_LAYER = 16
li = layers.index(INJECT_LAYER)
V = vectors[:, li, :].numpy()  # [C, D]
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
C = Vn @ Vn.T

# Reorder by Plutchik wheel sequence for visual coherence.
order = [cats.index(c) for c in PLUTCHIK_ORDER]
C_o = C[np.ix_(order, order)]
labels = [cats[i] for i in order]

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(C_o, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels)), labels)
ax.set_title(f'cos(v_a, v_b)  @ layer {INJECT_LAYER}')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{C_o[i,j]:+.2f}', ha='center', va='center',
                color='black', fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()

## 3. Emotion-code basis (NMF & PCA) — category loadings
Each category's mean Δ-activation projected onto the learned basis.

In [ ]:
basis = torch.load(EC / 'basis.pt', weights_only=False, map_location='cpu')
nmf_load = basis['nmf']['category_loadings'].numpy()   # [C, k]
pca_load = basis['pca']['category_loadings'].numpy()   # [C, k]
k = basis['k']
bcats = basis['categories']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, M, title in [(axes[0], nmf_load, f'NMF (k={k})'),
                     (axes[1], pca_load, f'PCA (k={k})')]:
    vmax = float(np.abs(M).max())
    im = ax.imshow(M, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(k), [f'b{j}' for j in range(k)])
    ax.set_yticks(range(len(bcats)), bcats)
    ax.set_xlabel('basis component')
    ax.set_title(f'{title} category loadings @ layer {basis["layer"]}')
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()

evr = np.array(basis['pca']['explained_variance_ratio'])
fig2, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(range(1, len(evr)+1), evr, color='#4F7BC7')
ax.set_xlabel('PCA component'); ax.set_ylabel('explained variance ratio')
ax.set_title(f'PCA spectrum (sum={evr.sum():.3f})')
fig2.tight_layout()

## 4. Layer probe sweep
Logistic regression on activations — which layer best linearly separates the 8 categories?

In [ ]:
ls = json.loads((RES / 'layer_sweep.json').read_text())
rows = ls.get('per_layer', ls)  # support either schema
if isinstance(rows, dict):
    rows = [{'layer': int(k), **v} for k, v in rows.items()]
df_ls = pd.DataFrame(rows).sort_values('layer')
best_layer = ls.get('best_layer')

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(df_ls['layer'], df_ls['val_acc'], marker='o', color='#4F7BC7')
if best_layer is not None:
    bl = df_ls[df_ls['layer'] == best_layer].iloc[0]
    ax.scatter([bl['layer']], [bl['val_acc']], s=160, facecolors='none',
               edgecolors='#D9462E', linewidth=2, label=f'best = layer {best_layer}')
    ax.legend()
ax.set_xlabel('hook layer'); ax.set_ylabel('val accuracy (8-way)')
ax.set_title('Linear probe accuracy across layers')
ax.set_xticks(df_ls['layer'])
fig.tight_layout()
df_ls

## 5. Steering shift accuracy — baseline vs α=+2
Generations classified by `j-hartmann/emotion-english-distilroberta-base` (Ekman 6 + neutral). `anticipation`/`trust` have no matching label and are omitted.

In [ ]:
df_shift = pd.read_csv(RES / 'shift_accuracy.csv')
df_shift = df_shift.dropna(subset=['shift_acc']).copy()
df_shift = df_shift.sort_values('shift_acc', ascending=False)

x = np.arange(len(df_shift))
w = 0.38
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(x - w/2, df_shift['baseline_acc'], w, label='baseline (α=0)', color='#B0B0B0')
ax.bar(x + w/2, df_shift['shift_acc'],    w, label='steered (α=+2)',
       color=[PLUTCHIK_COLORS[c] for c in df_shift['category']])
ax.set_xticks(x, df_shift['category'], rotation=20)
ax.set_ylabel('classifier accuracy on target class')
ax.set_title('Shift accuracy per emotion (Llama-3.1-8B, layer 16)')
ax.set_ylim(0, max(0.5, df_shift['shift_acc'].max() * 1.2))
ax.legend()
fig.tight_layout()
df_shift[['category','baseline_acc','shift_acc','delta']]

## 6. Monotonicity — Spearman ρ between α and target-class probability

In [ ]:
df_mono = pd.read_csv(RES / 'monotonicity.csv').dropna(subset=['rho'])
df_mono = df_mono.sort_values('rho', ascending=False)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.barh(df_mono['category'], df_mono['rho'],
        color=[PLUTCHIK_COLORS[c] for c in df_mono['category']])
ax.axvline(0.7, ls='--', color='#888', label='target ρ ≥ 0.7')
ax.set_xlabel('Spearman ρ (alpha_unit ↔ p(target class))')
ax.set_title('Steering monotonicity per emotion')
ax.legend()
ax.invert_yaxis()
fig.tight_layout()
df_mono[['category','rho','p_value']]

## 7. Perplexity guardrail — PPL(α) / PPL(0)
Largest α whose generation perplexity stays ≤ 2× baseline.

In [ ]:
df_ppl = pd.read_csv(RES / 'perplexity_alpha.csv')
base = df_ppl[df_ppl['alpha_unit'] == 0]['mean_ppl'].iloc[0]
df_ppl['ratio'] = df_ppl['mean_ppl'] / base

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cat in PLUTCHIK_ORDER:
    sub = df_ppl[df_ppl['category'] == cat]
    ax.plot(sub['alpha_unit'], sub['ratio'], marker='o',
            color=PLUTCHIK_COLORS[cat], label=cat, linewidth=1.6)
ax.axhline(2.0, ls='--', color='#666', label='ratio = 2× (guardrail)')
ax.set_yscale('log')
ax.set_xlabel('α (std-norm units)')
ax.set_ylabel('PPL(α) / PPL(0)   [log scale]')
ax.set_title('Perplexity vs steering strength')
ax.legend(ncol=2, fontsize=8, loc='upper left')
fig.tight_layout()

df_ppl_chosen = pd.read_csv(RES / 'perplexity_chosen.csv')
df_ppl_chosen

## 8. VAD readout — Ridge regression R²
EmoBank held-out (≈1.6k sentences), Llama layer 19 last-token residual.

In [ ]:
vad = torch.load(EC / 'vad_mapping.pt', weights_only=False, map_location='cpu')
r2 = vad['r2']

fig, ax = plt.subplots(figsize=(4.8, 3.2))
names = list(r2.keys())
vals = [r2[n] for n in names]
colors = ['#F4C53D', '#D9462E', '#4F7BC7']
ax.bar(names, vals, color=colors)
ax.axhline(0.5, ls='--', color='#666', label='target R² ≥ 0.5')
ax.set_ylabel('held-out R²')
ax.set_title(f'VAD regression @ layer {vad["layer"]}')
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=10)
ax.set_ylim(min(0, min(vals) - 0.05), max(0.7, max(vals) + 0.1))
ax.legend()
fig.tight_layout()
r2

## 9. Sample steered generations
First neutral prompt, all categories at α ∈ {-2, 0, +2}.

In [ ]:
df_gen = pd.read_parquet(RES / '_gen_cache.parquet')
subset = df_gen[(df_gen['prompt_id'] == 0) & (df_gen['alpha_unit'].isin([-2.0, 0.0, 2.0]))]
pivot = subset.pivot_table(index='category', columns='alpha_unit',
                           values='generation', aggfunc='first')
pivot = pivot.reindex(PLUTCHIK_ORDER)
prompt_text = df_gen[df_gen['prompt_id'] == 0]['prompt'].iloc[0]
print('PROMPT:', repr(prompt_text), '\n')
with pd.option_context('display.max_colwidth', 120):
    display(pivot)

## 10. Go / no-go scorecard

In [ ]:
shift_json = json.loads((RES / 'shift_accuracy.json').read_text().replace('NaN', 'null'))
mono_json  = json.loads((RES / 'monotonicity.json').read_text().replace('NaN', 'null'))
ppl_json   = json.loads((RES / 'perplexity_alpha.json').read_text().replace('NaN', 'null'))
vad_json   = json.loads((RES / 'vad_r2.json').read_text())

scorecard = pd.DataFrame([
    {'metric': 'shift accuracy (mean, α=+2)',
     'value': shift_json['mean_shift_acc'], 'target': 0.40},
    {'metric': 'monotonicity ρ (min over cats)',
     'value': float(np.nanmin([r['rho'] for r in mono_json['per_category'] if r['rho'] is not None])),
     'target': 0.70},
    {'metric': 'median max α @ ratio≤2',
     'value': ppl_json.get('median_max_alpha_unit', float(np.median(
         pd.read_csv(RES/'perplexity_chosen.csv')['max_alpha_unit']))),
     'target': 1.0},
    {'metric': 'VAD R² (min V/A/D)',
     'value': float(min(vad_json['r2'].values())), 'target': 0.50},
])
scorecard['pass'] = scorecard['value'] >= scorecard['target']
scorecard